## Access Images and Files from Drive

In [0]:
!pip install -q h5py pyyaml 
!pip install -U -q PyDrive
!pip install tflearn

    100% |████████████████████████████████| 102kB 2.5MB/s 
  Running setup.py bdist_wheel for tflearn ... - \ done
  Stored in directory: /root/.cache/pip/wheels/d0/f6/69/0ef3ee395aac2e5d15d89efd29a9a216f3c27767b43b72c006
Successfully built tflearn


In [0]:
import os
import numpy as np
# Set root directory of the project
ROOT_DIR = os.path.abspath('/Furniture')

# Directory to save logs and trained model
MODEL_DIR = os.path.join(ROOT_DIR, 'logs')

if not os.path.exists(ROOT_DIR):
    os.makedirs(ROOT_DIR)
os.chdir(ROOT_DIR)

# Create path /Furniture/Train_Dir
TRAIN_DIR = os.path.join(ROOT_DIR, 'Train_Dir')

if not os.path.exists(TRAIN_DIR):
    os.makedirs(TRAIN_DIR)
os.chdir(TRAIN_DIR)

In [0]:
# Download class images from drive to each class
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

os.chdir(TRAIN_DIR)
file_list = drive.ListFile({'q': "'root' in parents and trashed=false"}).GetList()

folder_name = ['bench.zip','cabinet_shelf.zip','chair.zip','sofa.zip','table.zip', 'TV_stand.zip']
for file in file_list:
    if (file['title'] in folder_name):  
        sub_folder = drive.CreateFile({'id': file['id']}) 
        sub_folder.GetContentFile(file['title'])
        
for i in os.listdir(TRAIN_DIR):
    !unzip -q -o $i

## Explore Data

In [0]:
!rm -rf '__MACOSX'
!rm -rf 'sofa.zip'
!rm -rf 'cabinet_shelf.zip'
!rm -rf 'TV_stand.zip'
!rm -rf 'bench.zip'
!rm -rf 'chair.zip'
!rm -rf 'table.zip'
os.listdir(TRAIN_DIR)


['bench', 'TV_stand', 'sofa', 'cabinet_shelf', 'table', 'chair']

## Data Preprocessing

In [0]:
import re
import cv2
import numpy as np
import os
import random
from tqdm import tqdm
import time

from PIL import Image
import matplotlib.pyplot as plt
%matplotlib inline
from __future__ import print_function
import keras
from keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

IMG_SIZE = 256
RANDOM_CROP_SIZE = 224
LR = 0.00001  # very small, better that 0.0001
BATCH_SIZE = 8
NUM_EPOCH = 2
NUM_CLASSES = 6
MOMENTUM = 0.9
SPLIT_RATE = 0.1
# RESCALE = 1./255

MODEL_NAME = 'furniture-{}-{}-{}.model'.format('augmented',LR,'resnet50')

Using TensorFlow backend.


In [0]:
# No data augmentation 
# generator = ImageDataGenerator(validation_split=0.1) # split to train and test set

# Add data augmentation
generator = ImageDataGenerator(validation_split=SPLIT_RATE,
      rotation_range=20,
#       width_shift_range=0.2,
#       height_shift_range=0.2,
      shear_range=0.2,
      zoom_range=0.2,
      horizontal_flip=True,
#       vertical_flip=True,
      fill_mode='nearest')
  
# Data Generator for Training data
train_generator = generator.flow_from_directory(TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
                                                        classes=['bench','cabinet_shelf','chair','sofa','table', 'TV stand'],
                                                         batch_size=BATCH_SIZE, class_mode='categorical', subset='training',
                                                         shuffle=True)
val_generator = generator.flow_from_directory(TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
                                                        classes=['bench','cabinet_shelf','chair','sofa','table', 'TV stand'],
                                                         batch_size=BATCH_SIZE, class_mode='categorical', subset='validation',
                                                         shuffle=False)  # in order to get val file name, shuffle needs to be True

Found 6361 images belonging to 6 classes.
Found 705 images belonging to 6 classes.


In [0]:
# ######### Crop is worse than non crop #########
# TRY - Add Random Cropping in ImageDataGenerator 
# ###############################################

def random_crop(img, random_crop_size):
    # Note: image_data_format is 'channel_last'
    assert img.shape[2] == 3
    height, width = img.shape[0], img.shape[1]
    dy, dx = random_crop_size
    x = np.random.randint(0, width - dx + 1)
    y = np.random.randint(0, height - dy + 1)
    return img[y:(y+dy), x:(x+dx), :]


def crop_generator(batches, crop_length):
    """Take as input a Keras ImageGen (Iterator) and generate random
    crops from the image batches generated by the original iterator.
    """
    while True:
        batch_x, batch_y = next(batches)
        batch_crops = np.zeros((batch_x.shape[0], crop_length, crop_length, 3))
        for i in range(batch_x.shape[0]):
            batch_crops[i] = random_crop(batch_x[i], (crop_length, crop_length))
        yield (batch_crops, batch_y)

In [0]:
train_crops = crop_generator(train_generator, RANDOM_CROP_SIZE)
val_crops = crop_generator(val_generator, RANDOM_CROP_SIZE)

In [0]:
#@title
# Visualization
imgs,labels = next(train_crops)

In [0]:
#@title
# may need to refine after adding augmentation
def plots(imgs, figsize=(18,9), rows=1, interp=False, titles=None):
    if type(imgs[0]) is np.ndarray:
        imgs = np.array(imgs).astype(np.uint8)
        if (imgs.shape[-1] !=3 ):
            imgs = imgs.transpose((0,2,3,1))
    f = plt.figure(figsize=figsize)
    cols = len(imgs)//rows if len(imgs)%2 == 0 else len(imgs)//rows+1
    for i in range(len(imgs)):
        sp = f.add_subplot(rows,cols,i+1)
        sp.axis('Off')
        if titles is not None:
            sp.set_title(titles[i],fontsize=16)
        plt.imshow(imgs[i],interpolation=None if interp else 'none')

In [0]:
#@title
plots(imgs,titles=labels)

##  Train the Model

In [0]:
# Transfer Learning Model - Resnet50
import tensorflow as tf
from tensorflow import keras
from tensorflow.python.keras.models import Sequential
from tensorflow.python.keras.layers import Dense, Flatten, GlobalAveragePooling2D
from IPython.display import SVG
from keras.utils.vis_utils import model_to_dot
from keras.utils import plot_model
import seaborn as sns
import matplotlib.pyplot as plt

from keras.applications import ResNet50
from keras import models
from keras import layers
from keras import optimizers  # Adam, RMSprop

# Load the Resnet50 model
resnet50_conv = ResNet50(weights='imagenet', include_top=False, input_shape=(RANDOM_CROP_SIZE, RANDOM_CROP_SIZE, 3))

# Freeze all the layers except the last 4 layers
for layer in resnet50_conv.layers[:-4]:
    layer.trainable = False

# Check the trainable status of the individual layers
for layer in resnet50_conv.layers:
    print(layer, layer.trainable)

# Create the model
model = models.Sequential()

# Add the vgg convolutional base model
model.add(resnet50_conv)

# Add new layers
model.add(layers.Flatten())
model.add(layers.Dense(1024, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(NUM_CLASSES, activation='softmax')) # num here is the class number

# compile the model, using a sigmoid optimized, the loss function as categorical crossentropy and the metric accuracy

# OPTION: sgd
model.compile(optimizer=optimizers.RMSprop(lr=LR), loss='categorical_crossentropy', metrics=['accuracy'])

# Show a summary of the model. Check the number of trainable parameters
model.summary()

/usr/local/lib/python3.6/dist-packages/keras_applications/resnet50.py:265: UserWarning: The output shape of `ResNet50(include_top=False)` has been changed since Keras 2.2.0.
  warnings.warn('The output shape of `ResNet50(include_top=False)` '


<keras.engine.input_layer.InputLayer object at 0x7fb83ea57898> False
<keras.layers.convolutional.ZeroPadding2D object at 0x7fb83ea57b70> False
<keras.layers.convolutional.Conv2D object at 0x7fb83ea57c50> False
<keras.layers.normalization.BatchNormalization object at 0x7fb83ea57cf8> False
<keras.layers.core.Activation object at 0x7fb83ea57fd0> False
<keras.layers.convolutional.ZeroPadding2D object at 0x7fb83ea50dd8> False
<keras.layers.pooling.MaxPooling2D object at 0x7fb83ea50b38> False
<keras.layers.convolutional.Conv2D object at 0x7fb83ea6b8d0> False
<keras.layers.normalization.BatchNormalization object at 0x7fb83e9a05f8> False
<keras.layers.core.Activation object at 0x7fb83e9a0f98> False
<keras.layers.convolutional.Conv2D object at 0x7fb83e9bfac8> False
<keras.layers.normalization.BatchNormalization object at 0x7fb83e95def0> False
<keras.layers.core.Activation object at 0x7fb83e913160> False
<keras.layers.convolutional.Conv2D object at 0x7fb83e840630> False
<keras.layers.convolution

In [0]:
CLASS_WEIGHT = {0: 3.0,  # bench
                1: 12.0, # cabinet/shelf
                2: 1.0,  # chair
               3: 0.5,   # sofa
               4: 1.0,   # table
               5: 5.}    # TV stand

In [0]:
#@title
# If you already have the model
MODEL_NAME = '/cp-0002.ckpt' 
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)
os.chdir(ROOT_DIR)

file_list = drive.ListFile({'q': "'root' in parents and trashed=false"}).GetList()
for file in file_list:
     if (file['title'] == MODEL_NAME):  
          file_id = file['id']
          # upload weights from google drive
          weight_file = drive.CreateFile({'id': file_id}) 
          weight_file.GetContentFile(MODEL_NAME)
          saved_model = model.load_weights(MODEL_NAME)
          

In [0]:
def extract_features_cnn(img_path):
    """Returns a normalized features vector for image path and model specified in parameters file """
    print('Using model', parameters.FEATURE_MODEL)
    img = image.load_img(img_path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    if model_extension == "vgg19":
        x = keras.applications.vgg19.preprocess_input(x)
    elif model_extension == "vgg16":
        x = keras.applications.vgg16.preprocess_input(x)
    elif model_extension == "resnet":
        x = keras.applications.resnet50.preprocess_input(x)
    else:
        print('Wrong model name')
    model_features = model.predict(x, batch_size=1)
    total_sum = sum(model_features[0])
    features_norm = np.array(
        [val / total_sum for val in model_features[0]], dtype=np.float32)
    if parameters.FEATURE_MODEL == "resnet":
        print("reshaping resnet")
        features_norm = features_norm.reshape(2048)
    return features_norm

## Build Similarity Search

In [0]:
model = resnet50_conv

In [0]:
os.chdir(TRAIN_DIR + '/sofa' + '/sofas')
!rm -rf 'style-search'

for i in os.listdir():
    img = load_img(i, target_size=(224, 224))
    x = img_to_array(img)
    x = np.expand_dims(x, axis=0)
    # print(x)    # each image shape: (1,224,224,3)
    x = keras.applications.resnet50.preprocess_input(x)
    model_features = model.predict(x, batch_size=1)   # conv shape: (7*7*2048)
    total_sum = sum(model_features[0])
    features_norm = np.array(
        [val / total_sum for val in model_features[0]], dtype=np.float32)
    features_norm = features_norm.reshape(2048,1*7*7)
    print(features_norm, features_norm.shape) # shape = (2048,49)

In [0]:
# Installing needed libraries

# https://opencv.org/
!apt-get -qq install -y libsm6 libxext6 && pip install -q -U opencv-python
# !wget -q https://github.com/creotiv/computer_vision/raw/master/resources.tar.gz -O resources.tar.gz
# !tar xzf resources.tar.gz
!rm -rf 'resources'
!rm -rf 'resources.tar.gz'

In [0]:
import cv2
import numpy as np
import scipy
from scipy.misc import imread
import pickle
import random
import os
import matplotlib.pyplot as plt

In [0]:
# Feature extractor
def extract_features(image_path, vector_size=32):
    image = imread(image_path, mode="RGB")
    try:
        # Using KAZE, cause SIFT, ORB and other was moved to additional module
        # which is adding addtional pain during install
        alg = cv2.KAZE_create()
        # Dinding image keypoints
        kps = alg.detect(image)
        # Getting first 32 of them. 
        # Number of keypoints is varies depend on image size and color pallet
        # Sorting them based on keypoint response value(bigger is better)
        kps = sorted(kps, key=lambda x: -x.response)[:vector_size]
        # computing descriptors vector
        kps, dsc = alg.compute(image, kps)
        # Flatten all of them in one big vector - our feature vector
        dsc = dsc.flatten()
        # Making descriptor of same size
        # Descriptor vector size is 64
        needed_size = (vector_size * 64)
        if dsc.size < needed_size:
            # if we have less the 32 descriptors then just adding zeros at the
            # end of our feature vector
            dsc = np.concatenate([dsc, np.zeros(needed_size - dsc.size)])
    except cv2.error as e:
        print('Error: ', e)
        return None

    return dsc


def batch_extractor(images_path, pickled_db_path="features.pck"):
    files = [os.path.join(images_path, p) for p in sorted(os.listdir(images_path))]

    result = {}
    for f in files:
        print('Extracting features from image %s' % f)
        name = f.split('/')[-1] # delete .lower()
        try:
            result[name] = extract_features(f)
        except:
            print(name)
            pass
    
    # saving all our feature vectors in pickled file
    with open(pickled_db_path, 'wb') as fp:  # open in binary mode
        pickle.dump(result, fp)

In [0]:
class Matcher(object):

    def __init__(self, pickled_db_path="features.pck"):
        with open(pickled_db_path,'rb') as fp:
            self.data = pickle.load(fp)
        self.names = []
        self.matrix = []
        for k, v in self.data.items():
            self.names.append(k)
            self.matrix.append(v)
        self.matrix = np.array(self.matrix)
        self.names = np.array(self.names)

    def cos_cdist(self, vector):
        # getting cosine distance between search image and images database
        v = vector.reshape(1, -1)
        return scipy.spatial.distance.cdist(self.matrix, v, 'cosine').reshape(-1)

    def match(self, image_path, topn=5):
        features = extract_features(image_path)
        img_distances = self.cos_cdist(features)
        # getting top 5 records
        nearest_ids = np.argsort(img_distances)[:topn].tolist()
        nearest_img_paths = self.names[nearest_ids].tolist()

        return nearest_img_paths, img_distances[nearest_ids].tolist()

In [0]:
def show_img(path):
    img = imread(path, mode="RGB")
    plt.imshow(img)
    plt.show()
    
def run():
    images_path = TRAIN_DIR + '/sofa' + '/sofas'
    files = [os.path.join(images_path, p) for p in sorted(os.listdir(images_path))]
    # getting 3 random images 
    sample = random.sample(files, 3)
    
    batch_extractor(images_path)

    ma = Matcher('features.pck')
    
    for s in sample:
        print('Query image ==========================================')
        show_img(s)
        names, match = ma.match(s, topn=3)
        print('Result images ========================================')
        for i in range(3):
            # we got cosine distance, less cosine distance between vectors
            # more they similar, thus we subtruct it from 1 to get match value
            print('Match %s' % (1-match[i]))
            show_img(os.path.join(images_path, names[i]))

run()